# LongFlow — score the head-v2 run on GPU

Runtime: **any GPU** (T4/L4 fine — scoring only, no VibeVoice). Reads
`headv2_eval.zip` from your **Drive root** (the train notebook puts it
there). Two parts: (1) held-out teacher-forced curves per checkpoint per
arm; (2) closed-loop GN8-protocol scoring against the pre-registered
criteria (NOTES "HV2 RUN PRE-REGISTRATION"). Whisper numbers are
content-survival only (GN7 instrument rule) — identity/listenability
verdicts belong to the graded sim curve and Josh's ear.


In [ ]:
# ===== COLD START — run me first, wait for READY =====
NOTEBOOK_VERSION = "Score head-v2 (GPU) v1.0 (2026-08-17)"
print(f"*** {NOTEBOOK_VERSION} ***")
!pip install -q faster-whisper jiwer whisper-normalizer speechbrain praat-parselmouth

import torch
assert torch.cuda.is_available(), "no GPU — pick a GPU runtime"
import glob, json, os, sys, zipfile
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1
from src.eval.metrics import clip_metrics, _ecapa, _normalizer, _whisper

from google.colab import drive
drive.mount("/content/drive")
candidates = glob.glob("/content/drive/MyDrive/headv2_eval*.zip")  # root only, NOT recursive
assert candidates, "no headv2_eval*.zip in Drive root — did the train notebook's cell 7 run?"
zip_path = candidates[0]
print(f"using {zip_path} ({os.path.getsize(zip_path)/1e9:.2f} GB)")
AUD = "/content/eval_audio"
os.makedirs(AUD, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(AUD)
print(f"extracted {len(os.listdir(AUD))} entries")
print("READY")


In [ ]:
# ===== Part 1: held-out curves per checkpoint per arm =====
with open(f"{AUD}/manifest.json") as f:
    manifest = json.load(f)
results = {"held_out": {}, "closed_loop": {}, "verdicts": {}}

for key in sorted(manifest["checkpoints"], key=lambda k: (int(k.split(":")[0]), k)):
    entries = manifest["checkpoints"][key]
    rows = []
    for e in entries:
        m = clip_metrics(f"{AUD}/{e['audio']}", e["text"],
                         f"{AUD}/{e['teacher_audio']}", device="cuda")
        rows.append({**m, "utt_id": e["utt_id"], "target_words": e["target_words"]})
    wers = sorted(r["wer"] for r in rows)
    sims = sorted(r["speaker_sim"] for r in rows)
    by_bin = {}
    for r in rows:
        by_bin.setdefault(r["target_words"], []).append(r["wer"])
    results["held_out"][key] = {
        "n": len(rows),
        "wer_median": wers[len(wers) // 2],
        "sim_median": sims[len(sims) // 2],
        "wer_by_bin": {b: sorted(v)[len(v) // 2] for b, v in sorted(by_bin.items())},
        "rows": rows,
    }
    o = results["held_out"][key]
    print(f"{key:>9}: n={o['n']:>3}  wer_med={o['wer_median']:.3f}  "
          f"sim_med={o['sim_median']:.3f}  by_bin={ {b: round(v,3) for b,v in o['wer_by_bin'].items()} }")

for arm in ("A", "B"):
    steps = sorted(int(k.split(":")[0]) for k in results["held_out"] if k.endswith(arm))
    if len(steps) >= 2:
        best = min(steps, key=lambda s: results["held_out"][f"{s}:{arm}"]["wer_median"])
        if best != steps[-1]:
            print(f"WATCH arm {arm}: best held-out WER at step {best}, not {steps[-1]} "
                  f"— July E3 overfit signature; consider {best} as operating checkpoint")


In [ ]:
# ===== Part 2: closed loop — GN8 protocol vs pre-registered criteria =====
import jiwer
import torchaudio.functional as taf

with open(f"{AUD}/headv2_report.json") as f:
    report = json.load(f)
WIN, HOP = 4.0, 2.0
ecapa = _ecapa("cuda")   # GPU — the CPU default is the 3-hour Mac mistake of 2026-08-17
n = _normalizer()

script_words = []
for line in report["cl_script"].splitlines():
    if ":" in line:
        line = line.split(":", 1)[1]
    script_words.append(line.strip())
SCRIPT = n(" ".join(w for w in script_words if w))
N_SCRIPT = len(SCRIPT.split())

def win_embs(path):
    x, sr = sf.read(path, dtype="float32")
    x16 = taf.resample(torch.from_numpy(x), sr, 16000).numpy()
    dur = len(x) / sr
    E, ts = [], []
    for i in range(int((dur - WIN) // HOP) + 1):
        seg = x16[int(i * HOP * 16000): int((i * HOP + WIN) * 16000)]
        if len(seg) < 16000:
            break
        emb = ecapa.encode_batch(torch.from_numpy(seg)[None].to("cuda"))[0, 0]
        E.append(emb.detach().cpu())
        ts.append(i * HOP)
    return torch.stack(E), np.array(ts), dur

def transcribe(path):
    segs, _ = _whisper("cuda").transcribe(str(path), language="en", beam_size=1)
    return " ".join((s.text or "").strip() for s in segs)

ref_E, _, _ = win_embs(f"{AUD}/t1_turnsplit_p0.wav")
ref = ref_E.median(0).values

def score(path):
    E, ts, dur = win_embs(path)
    sim = torch.nn.functional.cosine_similarity(E, ref[None], dim=-1).numpy()
    voice = sim >= 0.5
    horizon = 0.0
    for i in range(len(ts)):
        if voice[i]:
            horizon = ts[i] + WIN
    hyp = n(transcribe(path))
    nw = len(hyp.split())
    return {"duration_s": round(dur, 1),
            "wer_vs_script": round(jiwer.wer(SCRIPT, hyp) if hyp else 1.0, 3),
            "coverage_pct": round(100 * min(nw, N_SCRIPT) / N_SCRIPT, 1),
            "voice_pct": round(100 * float(voice.mean()), 1),
            "horizon_s": float(horizon),
            "sim_median": round(float(np.median(sim)), 3),
            "sim_final_third": round(float(np.median(sim[-max(1, len(sim) // 3):])), 3),
            "rate_wpm": round(60 * nw / dur, 1)}

for p in sorted(glob.glob(f"{AUD}/closed_loop/*.wav")):
    tag = os.path.basename(p)[:-4]
    results["closed_loop"][tag] = score(p)
    print(f"{tag}: {results['closed_loop'][tag]}", flush=True)

# ---- pre-registered verdict (NOTES, HV2 RUN PRE-REGISTRATION) ----
GN8_REF = {"wer_vs_script": 0.031, "voice_pct": 61.7, "sim_median": 0.522,
           "sim_final_third": 0.453, "horizon_s": 238.0}
c = results["closed_loop"]
a = [c.get("hv2_heun8_s0"), c.get("hv2_heun8_s1")]
b = [c.get("hv2ctl_cfg_heun8_s0"), c.get("hv2ctl_cfg_heun8_s1")]
if all(a):
    full = lambda r: r["horizon_s"] >= r["duration_s"] - 20
    if all(r["wer_vs_script"] > 0.15 or r["coverage_pct"] < 90 for r in a):
        v = "FAIL — dual-stream training hurt on both seeds; diagnose before any further spend"
    elif any(r["wer_vs_script"] > 0.15 or r["coverage_pct"] < 90 for r in a):
        v = "MIXED — one seed fails the content bar; per-seed variance, reroll before concluding"
    elif all(r["wer_vs_script"] <= 0.08 and r["sim_final_third"] >= 0.55 and full(r) for r in a):
        v = ("STRONG — identity erosion materially reduced vs GN8 ref; the training-time "
             "information gap was a main driver; stage-2 scope shrinks to the residual")
    elif all(r["wer_vs_script"] <= 0.08 for r in a):
        v = ("PARTIAL — content parity holds, identity gains below the strong bar; "
             "stage-2 on-policy carries identity as planned")
    else:
        v = "BETWEEN — content above 0.08 but below fail bar; read rows + listen before labeling"
    results["verdicts"]["arm_A"] = v
if all(b):
    drift = max(abs(b[0]["wer_vs_script"] - GN8_REF["wer_vs_script"]),
                abs(b[1]["wer_vs_script"] - GN8_REF["wer_vs_script"]))
    results["verdicts"]["arm_B_replication"] = (
        f"arm B vs GN8 July-head ref: max WER drift {drift:.3f} "
        + ("(within reseed floor 0.06 — baseline replicates)" if drift <= 0.06
           else "(EXCEEDS reseed floor — filtered-pool retrain moved the baseline; flag before comparing)"))
if "hv2_heun8_s0_sig02" in c:
    s02 = c["hv2_heun8_s0_sig02"]
    results["verdicts"]["sigma_conditioning"] = (
        f"sig02 (told): voice {s02['voice_pct']}% vs GN6 blind sigma=0.2 voice 0% — "
        + ("sigma-conditioning preserves identity where blind noise could not"
           if s02["voice_pct"] >= 20 else "no identity rescue from being told the noise level"))

print("\nVERDICTS:", json.dumps(results["verdicts"], indent=2))
with open("/content/headv2_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
with open("/content/drive/MyDrive/headv2_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("metrics on Drive root: headv2_metrics.json")
print("\nListening (never skipped): hv2_heun8_s0 vs hv2ctl_cfg_heun8_s0, and the sig02 arm.")
